# Tomahawk Rev 2.3.1 — Coupled Two-Temperature and Preferential Ion-Heating Screen

This Kaggle notebook updates the Tomahawk power balance with:

- electron–ion equipartition drag;
- explicit preferential ion heating P_aux_i;
- optional electron cooling/exhaust;
- alpha-heating partition and alpha-channeling scenarios;
- time-dependent Ti/Te integration;
- target-state power requirements for a 30 keV ion / 10 keV electron split;
- corrected ICRH frequency bookkeeping using the plasma field B0 and coil field Bpeak separately.

The result is a 0-D/profile-weighted thermal screen. It is not a validated kinetic, transport, nonlinear-MHD, RF-wave, neutral-beam, or alpha-channeling simulation.

In [1]:
# CELL 1 — Kaggle-safe imports and settings
import json, math, os, platform, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
os.environ.setdefault("MPLCONFIGDIR", "/tmp/tomahawk-rev231-two-temp-mpl")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = int(os.environ.get("TOMAHAWK_TWO_TEMP_SEED", "20260825"))
DT_S = float(os.environ.get("TOMAHAWK_TWO_TEMP_DT_S", "0.002"))
BURN_TIME_S = float(os.environ.get("TOMAHAWK_TWO_TEMP_BURN_S", "3.2"))
OUT = Path(os.environ.get("TOMAHAWK_TWO_TEMP_OUTPUT_ROOT", "/kaggle/working/tomahawk_rev231_two_temp"))
if not Path("/kaggle/working").exists():
    OUT = Path(os.environ.get("TOMAHAWK_TWO_TEMP_OUTPUT_ROOT", str(Path.cwd() / "tomahawk_rev231_two_temp")))
OUT = OUT.resolve()
OUT.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version(), "dt:", DT_S, "burn time:", BURN_TIME_S)
print("Output:", OUT)

Python: 3.12.13 dt: 0.002 burn time: 3.2
Output: /kaggle/working/tomahawk_rev231_two_temp


## 2. Inputs and field-frequency correction

The supplied design distinguishes a 35 T peak conductor field from an approximately 12–16 T plasma on-axis field. ICRH resonance inside the plasma must use B0, not the coil-surface peak field.

In [2]:
# CELL 2 — Physical inputs, profiles, and resonance frequencies
MU0 = 4.0e-7 * math.pi
KEV_TO_J = 1.0e3 * 1.602176634e-19
MEV_TO_J = 1.0e6 * 1.602176634e-19
AMU = 1.66053906660e-27
E_CHARGE = 1.602176634e-19
M_D, M_T = 2.01410177812 * AMU, 3.01604928199 * AMU
MU_D, MU_T = 2.01410177812, 3.01604928199
E_FUSION = 17.6 * MEV_TO_J
ALPHA_FRAC, NEUTRON_FRAC = 3.5 / 17.6, 14.1 / 17.6

R0_M, A_M, CAD_VOLUME_M3 = 7.5, 3.0, 950.0
B0_T, BPEAK_COIL_T = 14.0, 35.0
N_D0, N_T0 = 1.20e20, 1.20e20
GAMMA_N, GAMMA_T = 0.70, 0.80
TI_EDGE, TE_EDGE = 1.25, 1.25
Z_EFF = 1.0
TAU_E_S = 3.2
LASER_ABSORBED_W = 20.0e6
ALPHA_DEPOSITION = 0.90
ALPHA_TO_ION_BASELINE = 0.15

def ion_cyclotron_frequency_MHz(B_T, mass_amu):
    return E_CHARGE * B_T / (2.0 * math.pi * mass_amu * AMU) / 1.0e6

print(json.dumps({
    "D_fundamental_at_plasma_B0_MHz": ion_cyclotron_frequency_MHz(B0_T, MU_D),
    "T_fundamental_at_plasma_B0_MHz": ion_cyclotron_frequency_MHz(B0_T, MU_T),
    "D_fundamental_at_coil_Bpeak_MHz": ion_cyclotron_frequency_MHz(BPEAK_COIL_T, MU_D),
    "T_fundamental_at_coil_Bpeak_MHz": ion_cyclotron_frequency_MHz(BPEAK_COIL_T, MU_T),
}, indent=2))

{
  "D_fundamental_at_plasma_B0_MHz": 106.74020946498463,
  "T_fundamental_at_plasma_B0_MHz": 71.2805480215756,
  "D_fundamental_at_coil_Bpeak_MHz": 266.85052366246157,
  "T_fundamental_at_coil_Bpeak_MHz": 178.201370053939
}


## 3. Profile-weighted collisional and power model

The NRL/Spitzer thermal exchange rate is evaluated locally and integrated over the radial profile. Temperatures are in keV internally, while the collision coefficients use eV and densities in cm⁻³ where required. Reference: NRL Plasma Formulary, Thermal Equilibration, https://courses.grainger.illinois.edu/ece350/PlasmaFormulary.pdf.

In [3]:
# CELL 3 — D-T reactivity, bremsstrahlung, and electron-ion exchange
x, w = np.polynomial.legendre.leggauss(96)
RHO_Q, RHO_W = 0.5 * (x + 1.0), 0.5 * w * (x + 1.0)

def dt_reactivity(T_keV):
    T = np.clip(np.asarray(T_keV, dtype=float), 0.2, 100.0)
    BG, MRC2 = 34.3827, 1.124656e6
    C1,C2,C3,C4 = 1.17302e-9,1.51361e-2,7.51886e-2,4.60643e-3
    C5,C6,C7 = 1.35000e-2,-1.06750e-4,1.36600e-5
    num = T * (C2 + T * (C4 + T * C6))
    den = 1.0 + T * (C3 + T * (C5 + T * C7))
    theta = T / (1.0 - num / den)
    xi = (BG**2 / (4.0 * theta))**(1.0 / 3.0)
    return 1.0e-6 * C1 * theta * np.sqrt(xi / (MRC2 * T**3)) * np.exp(-3.0 * xi)

def coulomb_log(ne_cm3, Te_eV):
    value = 24.0 - np.log(np.maximum(np.sqrt(ne_cm3) / np.maximum(Te_eV, 1.0), 1.0))
    return float(np.clip(value, 10.0, 25.0))

def profile_metrics(Ti0_keV, Te0_keV, tau_E_s=TAU_E_S,
                    alpha_to_ion=ALPHA_TO_ION_BASELINE,
                    P_aux_i_W=0.0, P_e_cooling_W=0.0):
    reaction_i = brem_i = pressure_i = 0.0
    Wi = We = Pei = 0.0
    dWi_dTi = dWe_dTe = 0.0
    exchange_weight = 0.0
    for rrho, weight in zip(RHO_Q, RHO_W):
        ns = max(0.0, 1.0 - rrho**2)
        density_shape = ns**GAMMA_N
        temp_shape = ns**GAMMA_T
        nD = N_D0 * density_shape
        nT = N_T0 * density_shape
        ni = nD + nT
        ne = ni
        Ti = TI_EDGE + (Ti0_keV - TI_EDGE) * temp_shape
        Te = TE_EDGE + (Te0_keV - TE_EDGE) * temp_shape
        Te_eV = max(Te * 1.0e3, 200.0)
        lnL = coulomb_log(ne / 1.0e6, Te_eV)
        # NRL thermal equilibration coefficient, applied separately to D and T.
        nu_eD = 3.2e-9 * lnL * (nD / 1.0e6) / (MU_D * Te_eV**1.5)
        nu_eT = 3.2e-9 * lnL * (nT / 1.0e6) / (MU_T * Te_eV**1.5)
        nu_ei = nu_eD + nu_eT
        delta_T = Ti - Te
        reaction_i += weight * nD * nT * dt_reactivity(Ti)
        brem_i += weight * Z_EFF * ne * ni * np.sqrt(Te_eV)
        pressure_i += weight * (ni * Ti + ne * Te) * KEV_TO_J
        Wi += weight * 1.5 * ni * Ti * KEV_TO_J
        We += weight * 1.5 * ne * Te * KEV_TO_J
        dWi_dTi += weight * 1.5 * ni * temp_shape * KEV_TO_J
        dWe_dTe += weight * 1.5 * ne * temp_shape * KEV_TO_J
        Pei += weight * 1.5 * ne * delta_T * KEV_TO_J * nu_ei
        exchange_weight += weight * 1.5 * ne * abs(delta_T) * KEV_TO_J

    reaction_rate = CAD_VOLUME_M3 * reaction_i
    P_fusion = reaction_rate * E_FUSION
    P_alpha = P_fusion * ALPHA_FRAC
    P_alpha_dep = P_alpha * ALPHA_DEPOSITION
    P_alpha_i = P_alpha_dep * alpha_to_ion
    P_alpha_e = P_alpha_dep * (1.0 - alpha_to_ion)
    P_brem = CAD_VOLUME_M3 * 5.35e-37 * brem_i
    Wi *= CAD_VOLUME_M3
    We *= CAD_VOLUME_M3
    dWi_dTi *= CAD_VOLUME_M3
    dWe_dTe *= CAD_VOLUME_M3
    Pei *= CAD_VOLUME_M3
    P_trans_i = Wi / max(tau_E_s, 1.0e-9)
    P_trans_e = We / max(tau_E_s, 1.0e-9)
    delta_energy = exchange_weight * CAD_VOLUME_M3
    tau_ei_effective = abs(delta_energy / Pei) if abs(Pei) > 1.0 else float("inf")
    nu_ei_effective = 1.0 / tau_ei_effective if np.isfinite(tau_ei_effective) else 0.0

    P_aux_i_required_raw = P_trans_i + Pei - P_alpha_i
    P_e_cooling_required_raw = P_alpha_e + LASER_ABSORBED_W + Pei - P_brem - P_trans_e
    return {
        "Ti0_keV": Ti0_keV, "Te0_keV": Te0_keV,
        "P_fusion_W": P_fusion, "P_alpha_dep_W": P_alpha_dep,
        "P_alpha_i_W": P_alpha_i, "P_alpha_e_W": P_alpha_e,
        "P_brem_W": P_brem, "P_transport_i_W": P_trans_i, "P_transport_e_W": P_trans_e,
        "W_i_J": Wi, "W_e_J": We, "dWi_dTi_J_per_keV": dWi_dTi, "dWe_dTe_J_per_keV": dWe_dTe,
        "P_ei_W": Pei, "tau_ei_effective_s": tau_ei_effective,
        "nu_ei_effective_s-1": nu_ei_effective,
        "P_aux_i_required_raw_W": P_aux_i_required_raw,
        "P_aux_i_required_W": max(P_aux_i_required_raw, 0.0),
        "P_e_cooling_required_raw_W": P_e_cooling_required_raw,
        "P_e_cooling_required_W": max(P_e_cooling_required_raw, 0.0),
        "alpha_self_heating_ratio": P_alpha_dep / max(P_brem + P_trans_i + P_trans_e, 1.0),
        "net_power_balance_W": P_alpha_dep + P_aux_i_W + LASER_ABSORBED_W + Pei - P_brem - P_trans_i - P_trans_e - Pei - P_e_cooling_W,
        "P_aux_i_W": P_aux_i_W, "P_e_cooling_W": P_e_cooling_W,
    }

TARGET = profile_metrics(30.0, 10.0, TAU_E_S, ALPHA_TO_ION_BASELINE)
print(json.dumps({k: (v / 1e9 if k.endswith("_W") else v) for k, v in TARGET.items()
                  if k in ("P_fusion_W","P_alpha_dep_W","P_alpha_i_W","P_alpha_e_W","P_brem_W",
                           "P_transport_i_W","P_transport_e_W","P_ei_W",
                           "P_aux_i_required_W","P_e_cooling_required_W","tau_ei_effective_s",
                           "nu_ei_effective_s-1","alpha_self_heating_ratio")}, indent=2))

{
  "P_fusion_W": 7.973372754155541,
  "P_alpha_dep_W": 1.427052509976702,
  "P_alpha_i_W": 0.2140578764965053,
  "P_alpha_e_W": 1.2129946334801966,
  "P_brem_W": 1.0707764237982558,
  "P_transport_i_W": 0.20950815718904606,
  "P_transport_e_W": 0.07252205500112335,
  "P_ei_W": 2.6057189449359486,
  "tau_ei_effective_s": 0.16822824574126438,
  "nu_ei_effective_s-1": 5.944304986321996,
  "P_aux_i_required_W": 2.601169225628489,
  "P_e_cooling_required_W": 2.695415099616766,
  "alpha_self_heating_ratio": 1.054882842834393
}


## 4. Coupled time-dependent Ti/Te integration

The equations integrated are:

dWi/dt = P_alpha,i + P_aux,i − P_transport,i − P_ei

dWe/dt = P_alpha,e + P_aux,e + P_ei − P_brem − P_transport,e − P_e,cooling

P_ei is positive when Ti exceeds Te and is added to the electrons. The ion and electron temperatures are recovered from their profile-weighted thermal energies.

In [4]:
# CELL 4 — RK4 coupled two-temperature integrator
def rhs(y, scenario):
    Ti = max(float(y[0]), 0.2)
    Te = max(float(y[1]), 0.2)
    m = profile_metrics(Ti, Te, scenario["tau_E_s"], scenario["alpha_to_ion"],
                        scenario["P_aux_i_W"], scenario["P_e_cooling_W"])
    dWi = m["P_alpha_i_W"] + scenario["P_aux_i_W"] - m["P_transport_i_W"] - m["P_ei_W"]
    dWe = m["P_alpha_e_W"] + LASER_ABSORBED_W + m["P_ei_W"] - m["P_brem_W"] - m["P_transport_e_W"] - scenario["P_e_cooling_W"]
    return np.array([dWi / max(m["dWi_dTi_J_per_keV"], 1.0), dWe / max(m["dWe_dTe_J_per_keV"], 1.0)])

def integrate_scenario(name, Ti_start, Te_start, scenario):
    steps = int(math.ceil(BURN_TIME_S / DT_S))
    times = np.linspace(0.0, steps * DT_S, steps + 1)
    states = np.zeros((steps + 1, 2), dtype=float)
    states[0] = [Ti_start, Te_start]
    for k in range(steps):
        y = states[k]
        h = DT_S
        k1 = rhs(y, scenario)
        k2 = rhs(y + 0.5 * h * k1, scenario)
        k3 = rhs(y + 0.5 * h * k2, scenario)
        k4 = rhs(y + h * k3, scenario)
        states[k + 1] = np.maximum(y + h * (k1 + 2*k2 + 2*k3 + k4) / 6.0, 0.2)
    rows = []
    for t, (Ti, Te) in zip(times, states):
        m = profile_metrics(Ti, Te, scenario["tau_E_s"], scenario["alpha_to_ion"],
                            scenario["P_aux_i_W"], scenario["P_e_cooling_W"])
        rows.append({"time_s": t, "Ti_keV": Ti, "Te_keV": Te, "delta_T_keV": Ti - Te,
                     **{k: v for k, v in m.items() if isinstance(v, (float, int, np.floating))}})
    result = pd.DataFrame(rows)
    result["scenario"] = name
    return result

PION_REQ = TARGET["P_aux_i_required_W"]
PECOOL_REQ = TARGET["P_e_cooling_required_W"]
SCENARIOS = {
    "no_preferential_ion_heat": {"tau_E_s": TAU_E_S, "alpha_to_ion": 0.15, "P_aux_i_W": 0.0, "P_e_cooling_W": 0.0},
    "ICRH_2GW": {"tau_E_s": TAU_E_S, "alpha_to_ion": 0.15, "P_aux_i_W": 2.0e9, "P_e_cooling_W": 0.0},
    "alpha_channeling_50pct_ICRH_1GW": {"tau_E_s": TAU_E_S, "alpha_to_ion": 0.50, "P_aux_i_W": 1.0e9, "P_e_cooling_W": 0.0},
    "target_hold_with_ion_heat_and_e_cooling": {"tau_E_s": TAU_E_S, "alpha_to_ion": 0.15,
                                                "P_aux_i_W": PION_REQ, "P_e_cooling_W": PECOOL_REQ},
}
TRAJECTORIES = pd.concat([integrate_scenario(k, 30.0, 10.0, v) for k, v in SCENARIOS.items()], ignore_index=True)
print(TRAJECTORIES.groupby("scenario").tail(1)[["scenario","Ti_keV","Te_keV","delta_T_keV","P_ei_W","alpha_self_heating_ratio"]].to_string(index=False))

                               scenario     Ti_keV    Te_keV  delta_T_keV       P_ei_W  alpha_self_heating_ratio
               no_preferential_ion_heat   0.200000  0.200000     0.000000 0.000000e+00                  0.000005
                               ICRH_2GW 149.127409 59.571796    89.555612 9.592416e+08                  0.610571
        alpha_channeling_50pct_ICRH_1GW 106.962638 40.753860    66.208778 1.217370e+09                  0.783683
target_hold_with_ion_heat_and_e_cooling  30.000000 10.000000    20.000000 2.605719e+09                  1.054883


## 5. Ion-heating requirement sweep

This sweep reports the continuous ion-heating power needed at the target state. It does not assume that ICRH or NBI can deliver the requested power; those systems require separate RF, beam-transport, shielding, and wall-load designs.

In [5]:
# CELL 5 — Auxiliary ion-heating and alpha-channeling sweep
rows = []
for alpha_i in np.linspace(0.0, 0.80, 9):
    m = profile_metrics(30.0, 10.0, TAU_E_S, alpha_i)
    rows.append({
        "alpha_to_ion_fraction": alpha_i,
        "P_ei_GW": m["P_ei_W"] / 1e9,
        "P_aux_i_required_GW": m["P_aux_i_required_W"] / 1e9,
        "P_e_cooling_required_GW": m["P_e_cooling_required_W"] / 1e9,
        "alpha_self_heating_ratio": m["alpha_self_heating_ratio"],
    })
REQUIREMENT_SWEEP = pd.DataFrame(rows)
REQUIREMENT_SWEEP.to_csv(OUT / "ion_heating_requirement_sweep.csv", index=False)
print(REQUIREMENT_SWEEP.to_string(index=False))

 alpha_to_ion_fraction  P_ei_GW  P_aux_i_required_GW  P_e_cooling_required_GW  alpha_self_heating_ratio
                   0.0 2.605719             2.815227                 2.909473                  1.054883
                   0.1 2.605719             2.672522                 2.766768                  1.054883
                   0.2 2.605719             2.529817                 2.624062                  1.054883
                   0.3 2.605719             2.387111                 2.481357                  1.054883
                   0.4 2.605719             2.244406                 2.338652                  1.054883
                   0.5 2.605719             2.101701                 2.195947                  1.054883
                   0.6 2.605719             1.958996                 2.053241                  1.054883
                   0.7 2.605719             1.816290                 1.910536                  1.054883
                   0.8 2.605719             1.673585            

## 6. Export plots, manifest, and ZIP bundle

In [6]:
# CELL 6 — Diagnostics and downloadable results
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
for name, group in TRAJECTORIES.groupby("scenario"):
    ax[0].plot(group.time_s, group.Ti_keV, label=name)
    ax[1].plot(group.time_s, group.Te_keV, label=name)
ax[0].set_title("Ion temperature trajectories"); ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("Ti (keV)")
ax[1].set_title("Electron temperature trajectories"); ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("Te (keV)")
for axy in ax:
    axy.grid(alpha=0.25); axy.legend(fontsize=7)
fig.savefig(OUT / "two_temperature_trajectories.png", dpi=170); plt.close(fig)

fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.plot(REQUIREMENT_SWEEP.alpha_to_ion_fraction, REQUIREMENT_SWEEP.P_aux_i_required_GW, marker="o", label="required ion heating")
ax.plot(REQUIREMENT_SWEEP.alpha_to_ion_fraction, REQUIREMENT_SWEEP.P_e_cooling_required_GW, marker="s", label="required electron cooling")
ax.set_xlabel("fraction of deposited alpha power sent to ions")
ax.set_ylabel("continuous power (GW/module)")
ax.set_title("Power requirement for the 30/10 keV target")
ax.grid(alpha=0.25); ax.legend()
fig.savefig(OUT / "auxiliary_power_requirements.png", dpi=170); plt.close(fig)

TRAJECTORIES.to_csv(OUT / "two_temperature_trajectories.csv", index=False)
REQUIREMENT_SWEEP.to_csv(OUT / "ion_heating_requirement_sweep.csv", index=False)
pd.DataFrame([TARGET]).to_csv(OUT / "target_power_balance.csv", index=False)
TARGET_JSON = {k: (float(v) if np.isfinite(v) else None) for k, v in TARGET.items() if isinstance(v, (float, int, np.floating))}
(OUT / "target_power_balance.json").write_text(json.dumps(TARGET_JSON, indent=2), encoding="utf-8")

MANIFEST = {
    "model_revision": "2.3.1-coupled-two-temperature-ion-heating",
    "status": "preliminary 0-D/profile-weighted thermal screen",
    "target": {"Ti0_keV": 30.0, "Te0_keV": 10.0, "tau_E_s": TAU_E_S},
    "target_power_balance": TARGET_JSON,
    "scenarios": list(SCENARIOS.keys()),
    "limitations": [
        "No 3-D kinetic, nonlinear MHD, RF wave, NBI beamline, alpha-channeling, or wall model.",
        "Spitzer/NRL exchange rate assumes weakly coupled near-Maxwellian plasma and an assigned Coulomb logarithm.",
        "ICRH/NBI powers are user inputs; hardware efficiency and access geometry are not validated.",
        "The 35 T value is treated as a coil peak field; plasma resonance uses the 14 T on-axis field."
    ],
}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
zip_path = OUT / "Tomahawk_Rev231_Coupled_TwoTemperature_Results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p != zip_path:
            z.write(p, p.relative_to(OUT))

print("\nFINAL TARGET REQUIREMENTS")
print(json.dumps({
    "tau_ei_effective_s": TARGET["tau_ei_effective_s"],
    "nu_ei_effective_s-1": TARGET["nu_ei_effective_s-1"],
    "P_ei_GW": TARGET["P_ei_W"] / 1e9,
    "P_aux_i_required_GW": TARGET["P_aux_i_required_W"] / 1e9,
    "P_e_cooling_required_GW": TARGET["P_e_cooling_required_W"] / 1e9,
    "result_bundle": str(zip_path),
}, indent=2))


FINAL TARGET REQUIREMENTS
{
  "tau_ei_effective_s": 0.16822824574126438,
  "nu_ei_effective_s-1": 5.944304986321996,
  "P_ei_GW": 2.6057189449359486,
  "P_aux_i_required_GW": 2.601169225628489,
  "P_e_cooling_required_GW": 2.695415099616766,
  "result_bundle": "/kaggle/working/tomahawk_rev231_two_temp/Tomahawk_Rev231_Coupled_TwoTemperature_Results.zip"
}


## Interpretation

A no-heating trajectory that collapses toward Ti ≈ Te is the expected collisional result, not a software failure. A trajectory that holds Ti ≈ 30 keV and Te ≈ 10 keV requires the reported continuous ion-heating and electron-cooling powers. Those powers are screening requirements, not evidence that an ICRH, NBI, or alpha-channeling system can physically deliver them.